In [ ]:
import pandas as pd
import pickle
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from scipy.stats import pearsonr
from statsmodels.stats.multitest import fdrcorrection
from functools import reduce
import matplotlib.pyplot as plt
import re

# --------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------
data_dir = '/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/'
phenotype_df_path = data_dir + 'phenotypes_diet_microbiome_new.pkl'
lists_path = data_dir + 'my_lists_new.pkl'

# Outputs
results_path = data_dir + 'phenotypes_ablation_results_df_new.pkl'
predictions_path = data_dir + 'phenotypes_ablation_predictions_new.pkl'
figure_path = data_dir + 'phenotypes_ablation_barplot_new.png'

# --------------------------------------------------------------------
# Load data
# --------------------------------------------------------------------
df = pd.read_pickle(phenotype_df_path)
initial_samples_num = df.shape[0]

with open(lists_path, 'rb') as f:
    base_features, diet_features, target_phenotypes, microbial_features = pickle.load(f)

# --------------------------------------------------------------------
# Sanitize feature names for LightGBM (no special JSON chars)
# --------------------------------------------------------------------
def sanitize_names_in_df(df, feature_lists):
    """
    Replace non-alphanumeric/underscore chars in feature names with '_'
    and update df + feature lists accordingly.
    """
    # Flatten list of lists -> set of all feature names to be sanitized
    all_feats = []
    for lst in feature_lists:
        all_feats.extend(lst)
    all_feats = list(dict.fromkeys(all_feats))  # preserve order, remove dups

    mapping = {}
    used_new_names = set(df.columns)  # to avoid collisions with existing names

    for old in all_feats:
        if old not in df.columns:
            # Might happen if some list elements are outdated; just skip
            continue
        # Replace anything that's not a-zA-Z0-9_ with '_'
        new = re.sub(r'[^0-9a-zA-Z_]', '_', old)

        # Ensure uniqueness
        if new in used_new_names and new != old:
            # Append a short hash to disambiguate
            suffix = abs(hash(old)) % 10000
            new = f"{new}_{suffix}"

        mapping[old] = new
        used_new_names.add(new)

    # Actually rename df
    df = df.rename(columns=mapping)

    # Update feature lists
    def remap_list(lst):
        return [mapping.get(col, col) for col in lst]

    base_new = remap_list(base_features)
    diet_new = remap_list(diet_features)
    micro_new = remap_list(microbial_features)

    return df, base_new, diet_new, micro_new, mapping

# Apply sanitization to base + diet + micro features
df, base_features, diet_features, microbial_features, name_mapping = sanitize_names_in_df(
    df,
    [base_features, diet_features, microbial_features]
)

# (target_phenotypes and RegistrationCode are untouched – they are not model features)

metrics = []
all_predictions = []

# --------------------------------------------------------------------
# Helper functions
# --------------------------------------------------------------------
def get_model(n_samples: int) -> LGBMRegressor:
    """Return a LightGBM regressor with different n_estimators by sample size."""
    if n_samples < 3000:
        return LGBMRegressor(
            objective='regression',
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.9,
            colsample_bytree=0.8,
            min_child_samples=5,
            n_jobs=8,
            verbose=-1
        )
    else:
        return LGBMRegressor(
            objective='regression',
            n_estimators=800,
            learning_rate=0.01,
            max_depth=3,
            subsample=0.9,
            colsample_bytree=0.8,
            min_child_samples=5,
            n_jobs=8,
            verbose=-1
        )

def get_oof_preds(X: pd.DataFrame, y: pd.Series) -> np.ndarray:
    """5-fold OOF predictions for a given feature matrix X and target y."""
    y_pred = np.zeros(len(y))
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    n_samples = len(y)

    for train_idx, test_idx in kf.split(X):
        model = get_model(n_samples)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        y_pred[test_idx] = model.predict(X.iloc[test_idx])

    return y_pred


In [ ]:
base_features

In [ ]:
diet_features

In [ ]:
microbial_features

In [ ]:
target_phenotypes

In [ ]:
df

In [ ]:
# --------------------------------------------------------------------
# Main loop: build 4 models per phenotype (ablation)
# --------------------------------------------------------------------

for target in target_phenotypes:
    print(f"\n=== Target: {target} ===")

    if target != 'fat_mass_index':
        continue

    if target not in df.columns:
        print(f"  [WARNING] Target {target} not found in df.columns, skipping.")
        continue

    # Keep only non-missing target rows
    df_target = df[df[target].notna()].copy()
    print(f'{df_target.shape[0]} / {initial_samples_num} samples left with non-NA values')

    if df_target.shape[0] < 20:
        print("  [WARNING] Too few samples, skipping.")
        continue

    y_true = df_target[target]

    # Feature sets
    X_base = df_target[base_features]

    X_base_diet = df_target[base_features + diet_features]

    print("X_base_diet -----------------------------")
    print(X_base_diet)

    X_base_micro = df_target[base_features + microbial_features]

    print("X_base_micro -----------------------------")
    print(X_base_micro)

    full_cols = list(dict.fromkeys(base_features + diet_features + microbial_features))
    X_full = df_target[full_cols]

    # OOF predictions for each model
    y_pred_base = get_oof_preds(X_base, y_true)
    y_pred_diet = get_oof_preds(X_base_diet, y_true)
    y_pred_micro = get_oof_preds(X_base_micro, y_true)
    y_pred_full = get_oof_preds(X_full, y_true)

    # Correlations and p-values
    r_base, p_base = pearsonr(y_true, y_pred_base)
    r_diet, p_diet = pearsonr(y_true, y_pred_diet)
    r_micro, p_micro = pearsonr(y_true, y_pred_micro)
    r_full, p_full = pearsonr(y_true, y_pred_full)

    # Store metrics
    metrics.append({
        'target': target,
        'r_base': r_base,
        'r_diet': r_diet,
        'r_micro': r_micro,
        'r_full': r_full,
        'delta_r_diet': r_diet - r_base,
        'delta_r_micro': r_micro - r_base,
        'delta_r_full': r_full - r_base,
        'p_base': max(p_base, 1e-308),
        'p_diet': max(p_diet, 1e-308),
        'p_micro': max(p_micro, 1e-308),
        'p_full': max(p_full, 1e-308)
    })

    # Store predictions (all 4 models) for potential downstream use
    all_predictions.append(pd.DataFrame({
        'RegistrationCode': df_target['RegistrationCode'],
        f'{target}_pred_base': y_pred_base,
        f'{target}_pred_diet': y_pred_diet,
        f'{target}_pred_micro': y_pred_micro,
        f'{target}_pred_full': y_pred_full,
    }))

# --------------------------------------------------------------------
# Merge predictions across targets
# --------------------------------------------------------------------
if len(all_predictions) > 0:
    final_df = reduce(lambda left, right: pd.merge(left, right, on='RegistrationCode', how='outer'),
                      all_predictions)

    # Add base features (age/sex/BMI etc.) for convenience
    info_cols = ['RegistrationCode'] + base_features
    info_cols = list(dict.fromkeys(info_cols))
    final_df = final_df.merge(df[info_cols], on='RegistrationCode', how='left')

    # Reorder columns: RegistrationCode, base_features, then predictions
    pred_cols = [c for c in final_df.columns if c not in ['RegistrationCode'] + base_features]
    final_df = final_df[['RegistrationCode'] + base_features + pred_cols]

    # # Save predictions
    final_df.to_pickle(path=predictions_path)
else:
    print("No predictions were generated (no valid targets?). Skipping saving predictions.")
    final_df = None

# --------------------------------------------------------------------
# Build metrics DataFrame + FDR + sorting
# --------------------------------------------------------------------
results_df = pd.DataFrame(metrics)

if not results_df.empty:
    # FDR corrections per model
    for col in ['p_base', 'p_diet', 'p_micro', 'p_full']:
        fdr_col = 'fdr_' + col.split('_')[1]
        results_df[fdr_col] = fdrcorrection(results_df[col])[1]

    # Reorder columns nicely
    results_df = results_df[[
        'target',
        'r_base', 'r_diet', 'r_micro', 'r_full',
        'delta_r_diet', 'delta_r_micro', 'delta_r_full',
        'p_base', 'p_diet', 'p_micro', 'p_full',
        'fdr_base', 'fdr_diet', 'fdr_micro', 'fdr_full'
    ]]

    # Sort by overall gain when using everything (full model vs base)
    results_df = results_df.sort_values('delta_r_full', ascending=False).reset_index(drop=True)

    # Save metrics
    results_df.to_pickle(path=results_path)

    # ----------------------------------------------------------------
    # Visualization: grouped barplot for top-N phenotypes
    # ----------------------------------------------------------------
    top_n = 15  # adjust if you want more/less in the figure
    plot_df = results_df.head(top_n).copy()

    x = np.arange(len(plot_df))  # phenotype indices
    width = 0.2

    plt.figure(figsize=(14, 6))
    plt.bar(x - 1.5 * width, plot_df['r_base'], width, label='Base (age/sex/BMI)')
    plt.bar(x - 0.5 * width, plot_df['r_diet'], width, label='Base + diet')
    plt.bar(x + 0.5 * width, plot_df['r_micro'], width, label='Base + microbiome')
    plt.bar(x + 1.5 * width, plot_df['r_full'], width, label='Base + diet + microbiome')

    plt.xticks(x, plot_df['target'], rotation=90)
    plt.ylabel('Pearson r (OOF)')
    plt.title('Ablation: predictive performance by feature set (top phenotypes)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(figure_path, dpi=300)
    plt.show()
else:
    print("results_df is empty, skipping FDR and plotting.")

In [ ]:
# For the target you’re inspecting, e.g. 'fat_mass_index'
target = "fat_mass_index"

df_target = df[df[target].notna()].copy()
vc = df_target["RegistrationCode"].value_counts()
print("n rows:", df_target.shape[0])
print("unique RegistrationCode:", vc.shape[0])
print("dup counts:", vc.value_counts().head())


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

# -------------------------------------------------
# SANITY CHECKS FOR LEAKAGE / WEIRDNESS
# -------------------------------------------------

ID_COL = "RegistrationCode"  # change if needed

all_features = base_features + diet_features + microbial_features
all_features = [f for f in all_features if f in df.columns]  # keep existing only
all_features = list(dict.fromkeys(all_features))  # unique, preserve order

print("=== BASIC INFO ===")
print("df shape:", df.shape)
print("n targets:", len(target_phenotypes))
print("n base:", len(base_features), 
      "n diet:", len(diet_features), 
      "n micro:", len(microbial_features))
print()

# 1. Overlap of feature lists and with targets
print("=== NAME OVERLAP CHECKS ===")
sets = {
    "base": set(base_features),
    "diet": set(diet_features),
    "micro": set(microbial_features),
}
for a, b in [("base", "diet"), ("base", "micro"), ("diet", "micro")]:
    inter = sets[a] & sets[b]
    print(f"{a} ∩ {b}: {len(inter)} overlap")
    if inter:
        print("  examples:", sorted(list(inter))[:10])

tset = set(target_phenotypes)
for name, s in sets.items():
    t_overlap = tset & s
    print(f"Targets present in {name}_features: {len(t_overlap)}")
    if t_overlap:
        print("  ->", sorted(list(t_overlap)))
print()

# 2. ID uniqueness
if ID_COL in df.columns:
    vc = df[ID_COL].value_counts()
    print("=== ID DUPLICATES ===")
    print("rows:", len(df), "unique IDs:", vc.size)
    print("max rows per ID:", vc.max())
    if vc.max() > 1:
        print("  IDs with >1 row (possible leakage in CV):")
        print(vc[vc > 1].head())
    print()
else:
    print(f"[WARN] {ID_COL} not in df.columns – skipping ID checks\n")

# 3. Check that KFold (as used in your code) does not mix the same ID
#    into both train and test within any fold (only matters if IDs repeat)
if ID_COL in df.columns and df[ID_COL].value_counts().max() > 1:
    print("=== KFold ID OVERLAP CHECK ===")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for i, (tr, te) in enumerate(kf.split(df)):
        ids_tr = set(df.iloc[tr][ID_COL])
        ids_te = set(df.iloc[te][ID_COL])
        overlap = ids_tr & ids_te
        print(f"Fold {i}: {len(overlap)} IDs in both train & test")
        if overlap:
            print("  examples:", list(overlap)[:5])
    print()

# 4. Exact duplicates: any feature column == a target column
print("=== FEATURES IDENTICAL TO TARGETS (exact duplicates) ===")
for t in target_phenotypes:
    if t not in df.columns:
        continue
    y = df[t]
    identical = []
    for f in all_features:
        if f not in df.columns:
            continue
        if y.equals(df[f]):
            identical.append(f)
    print(f"Target {t}: {len(identical)} identical feature columns")
    if identical:
        print("  ->", identical)
print()

# 5. Extremely high correlations |r| > 0.99 between targets and features
print("=== FEATURES WITH |r| > 0.99 TO ANY TARGET ===")
for t in target_phenotypes:
    if t not in df.columns:
        continue
    y = df[t]
    high_corr = []
    for f in all_features:
        if f not in df.columns:
            continue
        valid = y.notna() & df[f].notna()
        if valid.sum() < 50:  # skip tiny overlaps
            continue
        r = np.corrcoef(y[valid], df.loc[valid, f])[0, 1]
        if np.isfinite(r) and abs(r) > 0.99:
            high_corr.append((f, r))
    high_corr = sorted(high_corr, key=lambda x: -abs(x[1]))
    print(f"Target {t}: {len(high_corr)} features with |r|>0.99")
    for f, r in high_corr[:10]:
        print(f"  {f:40s} r={r:.4f}")
print()

# 6. Near-perfect duplicates between feature columns themselves
#    (can inflate importance/ensemble behaviour, not strict leakage but good to know)
print("=== PAIRS OF FEATURES WITH |r| > 0.999 BETWEEN THEM (sampled) ===")
# to keep it tractable, sample at most 300 features
feat_sample = all_features[:300]
data = df[feat_sample].dropna()
corr = data.corr()
pairs = []
for i, a in enumerate(feat_sample):
    for j in range(i+1, len(feat_sample)):
        b = feat_sample[j]
        r = corr.iloc[i, j]
        if abs(r) > 0.999:
            pairs.append((a, b, r))
print(f"Found {len(pairs)} nearly-identical feature pairs in first {len(feat_sample)} features.")
for a, b, r in pairs[:15]:
    print(f"  {a}  <->  {b}   r={r:.4f}")
print()

# 7. NA patterns (just sanity, not leakage)
print("=== TOP 10 FEATURES BY #NA ===")
na_counts = df[all_features].isna().sum().sort_values(ascending=False)
print(na_counts.head(10))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor
from scipy.stats import pearsonr

ID_COL = "RegistrationCode"   # change if needed

def get_model(n_samples: int) -> LGBMRegressor:
    if n_samples < 3000:
        return LGBMRegressor(
            objective='regression',
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.9,
            colsample_bytree=0.8,
            min_child_samples=5,
            n_jobs=8,
            verbose=-1
        )
    else:
        return LGBMRegressor(
            objective='regression',
            n_estimators=800,
            learning_rate=0.01,
            max_depth=3,
            subsample=0.9,
            colsample_bytree=0.8,
            min_child_samples=5,
            n_jobs=8,
            verbose=-1
        )

def oof_corr(X, y):
    """5-fold OOF Pearson r for (X, y)."""
    y = y.to_numpy()
    y_pred = np.zeros_like(y, dtype=float)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    n_samples = len(y)
    for tr, te in kf.split(X):
        model = get_model(n_samples)
        model.fit(X.iloc[tr], y[tr])
        y_pred[te] = model.predict(X.iloc[te])
    r, _ = pearsonr(y, y_pred)
    return r

def permutation_check(target, feature_set="micro", n_perm=50):
    """
    feature_set: 'base', 'diet', 'micro', or 'full'
    """
    # subset to non-missing target
    df_t = df[df[target].notna()].copy()
    y_true = df_t[target]

    if feature_set == "base":
        feats = base_features
    elif feature_set == "diet":
        feats = base_features + diet_features
    elif feature_set == "micro":
        feats = base_features + microbial_features
    elif feature_set == "full":
        feats = list(dict.fromkeys(base_features + diet_features + microbial_features))
    else:
        raise ValueError("feature_set must be one of: base, diet, micro, full")

    feats = [f for f in feats if f in df_t.columns]
    X = df_t[feats]

    print(f"\n=== Permutation sanity check: target = {target}, features = {feature_set} ===")
    print("Samples:", len(df_t), "Features:", X.shape[1])

    # real OOF r
    r_real = oof_corr(X, y_true)
    print(f"Real OOF r: {r_real:.3f}")

    # permutations
    r_perm = []
    for i in range(n_perm):
        y_perm = y_true.sample(frac=1.0, replace=False, random_state=1000 + i).reset_index(drop=True)
        X_perm = X.reset_index(drop=True)
        r_p = oof_corr(X_perm, y_perm)
        r_perm.append(r_p)

    r_perm = np.array(r_perm)
    print(f"Permutation mean r: {r_perm.mean():.3f}, sd: {r_perm.std():.3f}")
    bigger = np.mean(np.abs(r_perm) >= abs(r_real))
    print(f"Empirical p (|r_perm| >= |r_real|): {bigger:.4g}")

    # plot
    plt.figure(figsize=(5,4))
    plt.hist(r_perm, bins=15, alpha=0.7)
    plt.axvline(r_real, color="red", linewidth=2, label=f"real r = {r_real:.3f}")
    plt.xlabel("OOF Pearson r (permuted labels)")
    plt.ylabel("Count")
    plt.title(f"Permutation test: {target} ({feature_set})")
    plt.legend()
    plt.tight_layout()
    plt.show()

# --------------------------------------------------------------------
# Example: run for BMI and trunk fat with base+microbiome
# --------------------------------------------------------------------
permutation_check("bmi", feature_set="micro", n_perm=30)
permutation_check("fat_mass_index", feature_set="micro", n_perm=30)
